In [18]:
import threading
import time
import random
from pynq.overlays.base import BaseOverlay
import pynq.lib.rgbled as rgbled
base = BaseOverlay("base.bit")

In [22]:
#static for part A2.1
#eatDurationCounter = 40 #5sec total eat time
#napDurationCounter = 20 # 5sec total nap time

#eatDurationCounter = 16 #2sec total eat time
#napDurationCounter = 8 # 2sec total nap time

#eatDurationCounter = 8 #2sec total eat time
#napDurationCounter = 4 # 2sec total nap time

eatDurationCounter = 4 #2sec total eat time
napDurationCounter = 2 # 2sec total nap time

eatFrequency = 0.125 #faster blinking lets assume 125ms
napFrequency = 0.250 #slower blinking  lets assume 250ms

led4 = rgbled.RGBLED(4) #one LED is RGB led

#randomised for A2.2
#eatDurationCounter = random.randint(20, 40)
#napDurationCounter = random.randint(10, 20)

print("Total EatDuration: {}, Total napDurationCounter: {}"\
      .format(eatDurationCounter*eatFrequency, napDurationCounter*napFrequency)) 

def blink(t, d, n):
    btns = base.btns_gpio
    for i in range(t):
        if n == 4: #i.e. RGB led
            led4.write(0x2) #assuming 2nd bit is for green led
            time.sleep(d/5) #time it spends as ON
            led4.write(0x0)
            time.sleep((1-d)/5) #time it spends as OFF
        else:
            base.leds[n].toggle()
            time.sleep(d)

        if btns[0].read() != 0:
            return
    starve(n)

#starve or reset function
def starve(n):
    if n == 4:
        led4.write(0x0)
    else:
        base.leds[n].off()

#forkList is the list of forks & num is the led to blink or professor Id
def philosopher_t(forkList, num):
    
    fork1AvailableStatus = False
    fork2AvailableStatus = False
    firstAcquiredFork = None
    
    btns = base.btns_gpio

    while btns[0].read() == 0: # this will be while loop until button is press

        # iterate through the forklist and try to acquire two fork
        # if one fork is available and other is not, release the one you have collected & go in STARVE mode
        # if both are available acquire them one by one and go to EAT mode for certain duration, release them and goto NAP mode
        # if none are available then also go to STARVE mode
        # in starve mode keep checking for fork availability on every 0.01 sec i.e. 10 milisecond

        time.sleep(0.01) #to handle button read input
        for fork in forkList:
            if fork1AvailableStatus == False:
                fork1AvailableStatus = fork.acquire(False)
                if fork1AvailableStatus:
                    #saving the fork variable to release them later
                    firstAcquiredFork = fork
                else:
                    print("Philosopher {} didn't get 1st fork, hence starving".format(num))
                    starve(num)
                    time.sleep(0.01)                    
            #if fork1 was acquired in last run, try to acquire fork2 now
            else:
                fork2AvailableStatus = fork.acquire(False)
                if fork2AvailableStatus:
                    #fork2 is also acquired, go to eat and once come out, release locks & go to nap
                    print("Philosopher {} has the forks, going to eat".format(num))
                    blink(eatDurationCounter,eatFrequency,num)
                    #release fork1 & fork2
                    fork.release()
                    firstAcquiredFork.release()
                    fork1AvailableStatus = False
                    fork2AvailableStatus = False
                    time.sleep(0.01)
                    print("Philosopher {} released forks, going to nap".format(num))
                    blink(napDurationCounter,napFrequency,num)
                    time.sleep(0.01)
                else:
                    #fork1 was acquired but not fork2, hence release fork1 & go in starve mode
                    firstAcquiredFork.release()
                    fork1AvailableStatus = False
                    print("Philosopher {} didn't get 2nd fork, hence starving".format(num))
                    starve(num)
                    time.sleep(0.01)
    print("interrupt received, stopping thread {}".format(num))

# Initialize and launch the threads
philosopherCount = 5

threads = []
forks = [] #list to maintain all forks i.e. lock associated with each fork

print("continue pressing button 0 to stop the program, until all LEDs are off")

#populate the fork list
for i in range(philosopherCount):
    fork = threading.Lock()
    forks.append(fork)

# create threads based on philosopher count and create 1 fork for each
for i in range(philosopherCount):
    t = threading.Thread(target=philosopher_t, args=(forks, i))
    threads.append(t)
    t.start()

for t in threads:
    name = t.getName()
    t.join()
    print('{} joined'.format(name))
    for i in range(philosopherCount):
        starve(i) #shutting down all LEDs

Total EatDuration: 0.5, Total napDurationCounter: 0.5
continue pressing button 0 to stop the program, until all LEDs are off
Philosopher 0 has the forks, going to eat
Philosopher 1 didn't get 1st fork, hence starving
Philosopher 2 didn't get 1st fork, hence starving
Philosopher 3 didn't get 1st fork, hence starving
Philosopher 4 didn't get 1st fork, hence starving
Philosopher 1 didn't get 1st fork, hence starving
Philosopher 2 didn't get 1st fork, hence starving
Philosopher 3 didn't get 1st fork, hence starving
Philosopher 4 didn't get 1st fork, hence starving
Philosopher 1 has the forks, going to eat
Philosopher 2 didn't get 1st fork, hence starving
Philosopher 3 didn't get 1st fork, hence starving
Philosopher 4 didn't get 1st fork, hence starving
Philosopher 2 didn't get 1st fork, hence starving
Philosopher 3 didn't get 1st fork, hence starving
Philosopher 4 didn't get 1st fork, hence starving
Philosopher 3 didn't get 1st fork, hence starving
Philosopher 4 didn't get 1st fork, hence 

/tmp/ipykernel_932/3652047209.py:122: DeprecationWarning: getName() is deprecated, get the name attribute instead
  name = t.getName()


Philosopher 2 didn't get 1st fork, hence starving
Philosopher 4 didn't get 1st fork, hence starving
Philosopher 3 didn't get 1st fork, hence starving
Philosopher 4 didn't get 1st fork, hence starving
Philosopher 3 didn't get 1st fork, hence starving
Philosopher 2 didn't get 2nd fork, hence starving
Philosopher 4 didn't get 1st fork, hence starving
Philosopher 2 didn't get 1st fork, hence starving
Philosopher 3 didn't get 2nd fork, hence starving
Philosopher 2 didn't get 1st fork, hence starving
Philosopher 3 didn't get 1st fork, hence starving
Philosopher 4 didn't get 2nd fork, hence starving
Philosopher 2 didn't get 1st fork, hence starving
Philosopher 3 didn't get 1st fork, hence starving
Philosopher 4 didn't get 1st fork, hence starving
Philosopher 3 didn't get 1st fork, hence starving
Philosopher 4 didn't get 1st fork, hence starving
Philosopher 2 didn't get 2nd fork, hence starving
Philosopher 4 didn't get 1st fork, hence starving
Philosopher 2 didn't get 1st fork, hence starving
